A **windowing operation** performs an aggregation (sum, mean,...) over a sliding partition of values.<br/>
The windows are comprised by **looking back** the length of the window (here 2) from the current observation.

In [1]:
import pandas as pd
import numpy as np

s = pd.Series(range(5))
s.rolling(window=2).sum()

0    NaN
1    1.0
2    3.0
3    5.0
4    7.0
dtype: float64

In [2]:
for window in s.rolling(window=2):
    print(window)

0    0
dtype: int64
0    0
1    1
dtype: int64
1    1
2    2
dtype: int64
2    2
3    3
dtype: int64
3    3
4    4
dtype: int64


# Overview

<dl>
    <dt style='font-weight:bold;color:green;'>Rolling window</dt>
    <dd>Sliding window over the values. Can be fixed or variable.</dd>
    <dt style='font-weight:bold;color:green;'>Weighted window</dt>
    <dd>Weighted, non-rectangular window from <b>scipy.signal</b> library.</dd>
    <dt style='font-weight:bold;color:green;'>Expanding window</dt>
    <dd>Accumulating window over the values.</dd>
    <dt style='font-weight:bold;color:green;'>Exponentially Weighted window</dt>
    <dd>Accumulating & exponentially weighted window over the values.</dd>
</dl>

<pre>
df.rolling(window,  # the size of the sliding window
           center,  # when True the label is at the center of the window; default is False
           closed,  # controls the inclusion of the interval endpoints in the rolling window
           )
</pre>

In [3]:
s = pd.Series(range(5), index=pd.date_range("2020-01-01", periods=5, freq="1D"))
s

2020-01-01    0
2020-01-02    1
2020-01-03    2
2020-01-04    3
2020-01-05    4
Freq: D, dtype: int64

In [4]:
# Window specified using a time offset (2 days)
s.rolling(window="2D").sum()

2020-01-01    0.0
2020-01-02    1.0
2020-01-03    3.0
2020-01-04    5.0
2020-01-05    7.0
Freq: D, dtype: float64

In [5]:
# Chaining a groupby operation, with a windowing operation.
# The windowing is performed for each group.
df = pd.DataFrame({"A": ["a", "b", "a", "b", "a"], "B": range(5)})
df

,A,B
0,a,0
1,b,1
2,a,2
3,b,3
4,a,4


In [6]:
df.groupby("A").expanding().sum()

B
A       
a 0  0.0
  2  2.0
  4  6.0
b 1  1.0
  3  4.0

#### Weighted Mean Calculation

Below columns 0 and 1 are the measurements and column 2 is a column with the weights

In [7]:
def weighted_mean(x: pd.DataFrame):
    arr = np.ones((1, x.shape[1]))
    arr[:, :2] = (x[:, :2] * x[:, 2]).sum(axis=0) / x[:, 2].sum()

    return arr


df = pd.DataFrame(
    [
        [1, 2, 0.6],
        [2, 3, 0.4],
        [3, 4, 0.2],
        [4, 5, 0.7],
    ]
)

df

,0,1,2
0,1,2,0.6
1,2,3,0.4
2,3,4,0.2
3,4,5,0.7


In [8]:
# method="table" executes the rolling operation over the entire table
# min_periods min number of measurements in the window, in order to return a value (else NaN)
# raw=True : ndarray will be feeded to the func (instead of each row/column separately)
# fmt: off
(
    df.rolling(2, method="table", min_periods=0)
      .apply(weighted_mean, raw=True, engine="numba")
)

,0,1,2
0,1.000000,2.000000,1.0
1,1.800000,2.000000,1.0
2,3.333333,2.333333,1.0
3,1.555556,7.000000,1.0


In [9]:
df = pd.DataFrame(
    [
        [1, 2, 0.6],
        [2, 3, 0.4],
        [3, 4, 0.2],
        [4, 5, 0.7],
    ]
)

df.ewm(0.5).mean()

,0,1,2
0,1.000000,2.000000,0.600000
1,1.750000,2.750000,0.450000
2,2.615385,3.615385,0.276923
3,3.550000,4.550000,0.562500


In [10]:
online_ewm = df.head(2).ewm(0.5).online()
online_ewm.mean()

,0,1,2
0,1.00,2.00,0.60
1,1.75,2.75,0.45


In [11]:
online_ewm.mean(update=df.tail(1))

,0,1,2
3,3.307692,4.307692,0.623077


In [12]:
s = pd.Series([np.nan, 1, 2, np.nan, np.nan, 3])
s.rolling(window=3, min_periods=1).sum()

0    NaN
1    1.0
2    3.0
3    3.0
4    2.0
5    3.0
dtype: float64

In [13]:
df = pd.DataFrame({"A": range(5), "B": range(10, 15)})
df

,A,B
0,0,10
1,1,11
2,2,12
3,3,13
4,4,14


In [14]:
df.expanding().agg(["sum", "mean", "std"])

A                    B                
    sum mean       std   sum  mean       std
0   0.0  0.0       NaN  10.0  10.0       NaN
1   1.0  0.5  0.707107  21.0  10.5  0.707107
2   3.0  1.0  1.000000  33.0  11.0  1.000000
3   6.0  1.5  1.290994  46.0  11.5  1.290994
4  10.0  2.0  1.581139  60.0  12.0  1.581139

# Rolling Window

In [15]:
# For a time-based offset, the time-based index must be monotonic.
times = ["2020-01-01", "2020-01-03", "2020-01-04", "2020-01-05", "2020-01-29"]
s = pd.Series(range(5), index=pd.DatetimeIndex(times))

s

2020-01-01    0
2020-01-03    1
2020-01-04    2
2020-01-05    3
2020-01-29    4
dtype: int64

In [16]:
# Window with 2 observations
s.rolling(window=2).sum()

2020-01-01    NaN
2020-01-03    1.0
2020-01-04    3.0
2020-01-05    5.0
2020-01-29    7.0
dtype: float64

In [17]:
# Window with 2 days worth of observations
s.rolling(window="2D").sum()

2020-01-01    0.0
2020-01-03    1.0
2020-01-04    3.0
2020-01-05    5.0
2020-01-29    4.0
dtype: float64

## Centering Windows

In [18]:
# By default, the labels are set to the right side of the window:
s = pd.Series(range(10))
s.rolling(window=5).mean()

0    NaN
1    NaN
2    NaN
3    NaN
4    2.0
5    3.0
6    4.0
7    5.0
8    6.0
9    7.0
dtype: float64

In [19]:
# To set the labels at the center of the window:
s.rolling(window=5, center=True).mean()

0    NaN
1    NaN
2    2.0
3    3.0
4    4.0
5    5.0
6    6.0
7    7.0
8    NaN
9    NaN
dtype: float64

In [20]:
# Also for datetime indices:
df = pd.DataFrame({"A": [0, 1, 2, 3, 4]}, index=pd.date_range("2020", periods=5, freq="1D"))
df

,A
2020-01-01,0
2020-01-02,1
2020-01-03,2
2020-01-04,3
2020-01-05,4


In [21]:
df.rolling(window="2D", center=False).mean()

,A
2020-01-01,0.0
2020-01-02,0.5
2020-01-03,1.5
2020-01-04,2.5
2020-01-05,3.5


In [22]:
df.rolling(window="2D", center=True).mean()

,A
2020-01-01,0.5
2020-01-02,1.5
2020-01-03,2.5
2020-01-04,3.5
2020-01-05,4.0


## Rolling Window Endpoints

In [23]:
df = pd.DataFrame(
    {"x": 1},
    index=[
        pd.Timestamp("20130101 09:00:01"),
        pd.Timestamp("20130101 09:00:02"),
        pd.Timestamp("20130101 09:00:03"),
        pd.Timestamp("20130101 09:00:04"),
        pd.Timestamp("20130101 09:00:06"),
    ],
)

In [24]:
# Default behavior: right endpoint is closed
df["right"] = df.rolling(window="2s", closed="right").x.sum()

In [25]:
# Close both endpoints
df["both"] = df.rolling(window="2s", closed="both").x.sum()

In [26]:
df["left"] = df.rolling(window="2s", closed="left").x.sum()

In [27]:
df["neither"] = df.rolling(window="2s", closed="neither").x.sum()
df

,x,right,both,left,neither
2013-01-01 09:00:01,1,1.0,1.0,NaN,NaN
2013-01-01 09:00:02,1,2.0,2.0,1.0,1.0
2013-01-01 09:00:03,1,2.0,3.0,2.0,1.0
2013-01-01 09:00:04,1,2.0,3.0,2.0,1.0
2013-01-01 09:00:06,1,1.0,2.0,1.0,NaN


## Custom Window Rolling

In [28]:
# Suppose we want to create an expanding window where use_expanding is True
# otherwise a window of size 1.
use_expanding = [True, False, True, False, True]
df = pd.DataFrame({"values": range(5)})

from pandas.api.indexers import BaseIndexer


class CustomIndexer(BaseIndexer):
    def get_window_bounds(self, num_values, min_periods, center, closed=None, step=None):
        start = np.empty(num_values, dtype=np.int64)
        end = np.empty(num_values, dtype=np.int64)

        for i in range(num_values):
            if self.use_expanding[i]:
                start[i] = 0
                end[i] = i + 1
            else:
                start[i] = i
                end[i] = i + self.window_size

        return start, end


indexer = CustomIndexer(window_size=1, use_expanding=use_expanding)
df.rolling(indexer).sum()

,values
0,0.0
1,1.0
2,3.0
3,3.0
4,10.0


In [29]:
# Rolling operations over a non-fixed offset (e.g. a BusinessDay)
from pandas.api.indexers import VariableOffsetWindowIndexer

df = pd.DataFrame(range(10), index=pd.date_range("2020", periods=10))
offset = pd.offsets.BDay(1)
indexer = VariableOffsetWindowIndexer(index=df.index, offset=offset)

df

,0
2020-01-01,0
2020-01-02,1
2020-01-03,2
2020-01-04,3
2020-01-05,4
2020-01-06,5
2020-01-07,6
2020-01-08,7
2020-01-09,8
2020-01-10,9


In [30]:
df.rolling(indexer).sum()

,0
2020-01-01,0.0
2020-01-02,1.0
2020-01-03,2.0
2020-01-04,3.0
2020-01-05,7.0
2020-01-06,12.0
2020-01-07,6.0
2020-01-08,7.0
2020-01-09,8.0
2020-01-10,9.0


In [31]:
# A closed fixed-width forward-looking rolling window
from pandas.api.indexers import FixedForwardWindowIndexer

indexer = FixedForwardWindowIndexer(window_size=2)
df.rolling(indexer, min_periods=1).sum()

,0
2020-01-01,1.0
2020-01-02,3.0
2020-01-03,5.0
2020-01-04,7.0
2020-01-05,9.0
2020-01-06,11.0
2020-01-07,13.0
2020-01-08,15.0
2020-01-09,17.0
2020-01-10,9.0


## Rolling apply

<pre>
apply(func, raw=True)   passes the window as an ndarray
apply(func, raw=False)  passes the window as a Series

np.fabs(x)              returns the absolute values of the elements in x, element-wise
</pre>

In [34]:
def mad(x):
    return np.fabs(x - x.mean()).mean()


s = pd.Series(range(10))
s.rolling(window=4).apply(mad, raw=True)

0    NaN
1    NaN
2    NaN
3    1.0
4    1.0
5    1.0
6    1.0
7    1.0
8    1.0
9    1.0
dtype: float64

## Numba Engine

## Binary Window Functions

To compute moving window statistics, use `cov()` and `corr()`.<br/>
Can be used in various combinations: <br/>
- Series/Series
- DataFrame/Series
- DataFrame/DataFrame

In [39]:
df = pd.DataFrame(
    np.random.randn(10, 4),
    index=pd.date_range("2020-01-01", periods=10),
    columns=list("ABCD"),
)
df = df.cumsum()
df2 = df[:4]
df2.rolling(window=2).corr(df2["B"])

,A,B,C,D
2020-01-01,NaN,NaN,NaN,NaN
2020-01-02,1.0,1.0,1.0,1.0
2020-01-03,1.0,1.0,1.0,-1.0
2020-01-04,-1.0,1.0,-1.0,-1.0


## Computing Rolling Pairwise Covariances and Correlations

In [44]:
# fmt: off
covs=(
    df[["B","C","D"]]
    .rolling(window=4)
    .cov(df[["A","B","C"]],pairwise=True)
)

covs.tail(6)

B         C         D
2020-01-09 A -0.087491 -0.673239 -0.078552
           B  0.399259 -0.229227  0.427677
           C -0.229227  0.978797 -0.207505
2020-01-10 A  1.266685 -0.403966  0.641106
           B  2.412215 -0.593224  1.051369
           C -0.593224  0.738874 -0.498733

# Weighted Window

In [45]:
s = pd.Series(range(10))
s.rolling(window=5).mean()

0    NaN
1    NaN
2    NaN
3    NaN
4    2.0
5    3.0
6    4.0
7    5.0
8    6.0
9    7.0
dtype: float64

In [46]:
s.rolling(window=5, win_type="triang").mean()

0    NaN
1    NaN
2    NaN
3    NaN
4    2.0
5    3.0
6    4.0
7    5.0
8    6.0
9    7.0
dtype: float64

In [47]:
# Supplementary Scipy arguments passed in the aggregation function
s.rolling(window=5, win_type="gaussian").mean(std=0.1)

0    NaN
1    NaN
2    NaN
3    NaN
4    2.0
5    3.0
6    4.0
7    5.0
8    6.0
9    7.0
dtype: float64

# Expanding Window

Computes an aggregation statistic window-wise, with all the data available to that point in time

In [51]:
# The following are the same:
df = pd.DataFrame(range(5))
df.rolling(window=len(df), min_periods=1).mean()

,0
0,0.0
1,0.5
2,1.0
3,1.5
4,2.0


In [49]:
df.expanding(min_periods=1).mean()

,0
0,0.0
1,0.5
2,1.0
3,1.5
4,2.0
